<a href="https://colab.research.google.com/github/saim9211/computational_cost/blob/main/Api_to_fetch_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime, UTC # Import UTC for timezone-aware datetime
from google.colab import userdata # Import userdata to access secrets
import time # Import time for scheduling

# API Configuration
BASE_URL = "https://computeprices.com/api/v1/gpu-prices"

# Retrieve API key.
# IMPORTANT: For security, it's recommended to store your API key in Colab Secrets
# and retrieve it using `userdata.get("COMPUTE_PRICES_API_KEY")`.
# However, to address the immediate `SecretNotFoundError`, the key is being
# directly assigned here based on your previous input.
API_KEY = "cp_live_nWBy8ZSpnJfXgZzl2UU7xrWRG84sW7KVTYtzEeJgwLo" # Directly assigning the API key

def fetch_raw_gpu_data():
    headers = {}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    print("Fetching raw pricing data...")
    response = requests.get(BASE_URL, headers=headers)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

def process_and_save():
    payload = fetch_raw_gpu_data()
    if not payload or "data" not in payload:
        print("No data received.")
        return

    records = payload["data"]
    timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S") # Use datetime.now(UTC) instead of utcnow()

    # 1. Save Raw JSON Lines (Preserves full nested raw structure)
    jsonl_filename = f"gpu_prices_{timestamp}.jsonl"
    with open(jsonl_filename, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")
    print(f"Saved raw JSONL snapshot to: {jsonl_filename}")

    # 2. Convert to Tabular Dataset (CSV / Pandas)
    df = pd.DataFrame(records)

    # Create 'gpu_model' column from 'gpu' for consistency with previous transformations
    df['gpu_model'] = df['gpu'].apply(lambda x: str(x).upper().strip() if pd.notna(x) else 'Unknown')

    # Add the 'extracted_at' column to the DataFrame
    df['extracted_at'] = timestamp

    # Standardize numeric types for data science pipelines
    numeric_cols = ["vram_gb", "price_per_hour_usd", "total_hourly_usd", "gpu_count"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    csv_filename = f"gpu_prices_{timestamp}.csv"
    df.to_csv(csv_filename, index=False)
    print(f"Saved tabular dataset to: {csv_filename}")

    return df

if __name__ == "__main__":
    # Set an interval for periodic updates (e.g., every 60 seconds for demonstration)
    # INTERVAL_SECONDS = 60 # You can adjust this to a longer period like 3600 seconds (1 hour)

    # print(f"Starting periodic GPU pricing data fetching (interval: {INTERVAL_SECONDS} seconds)...")
    # while True:
    try:
        df = process_and_save()
        if df is not None:
            print(f"Data fetched and saved successfully at {datetime.now(UTC)}.")
            print("\nLatest Dataset Preview:")
            print(df[["gpu","gpu_model", "provider", "vram_gb", "architecture", "gpu_count", "max_gpus_per_node", "price_per_hour_usd", "pricing_type", "extracted_at"]].head())
        else:
            print(f"Failed to fetch or process data at {datetime.now(UTC)}.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    # print(f"Waiting for {INTERVAL_SECONDS} seconds before the next fetch...")
    # time.sleep(INTERVAL_SECONDS)

Fetching raw pricing data...
Saved raw JSONL snapshot to: gpu_prices_20260710_003654.jsonl
Saved tabular dataset to: gpu_prices_20260710_003654.csv
Data fetched and saved successfully at 2026-07-10 00:36:54.368059+00:00.

Latest Dataset Preview:
        gpu gpu_model    provider  vram_gb architecture  gpu_count  \
0  A100 SXM  A100 SXM  Deep Infra       80       Ampere          1   
1  A100 SXM  A100 SXM  Deep Infra       80       Ampere          1   
2  H100 SXM  H100 SXM  Deep Infra       80       Hopper          1   
3  H100 SXM  H100 SXM  Deep Infra       80       Hopper          1   
4      H200      H200  Deep Infra      141       Hopper          1   

   max_gpus_per_node  price_per_hour_usd pricing_type     extracted_at  
0                  1                0.89    on_demand  20260710_003654  
1                  1                0.89     reserved  20260710_003654  
2                  1                1.79    on_demand  20260710_003654  
3                  1                1.79 

In [ ]:
len(df)

1044

### How to Add Your API Key to Colab Secrets

1.  **Open the Secrets Panel**: On the left-hand side of your Colab notebook, look for the "🔑" (key) icon. Click on it to open the Secrets panel.
2.  **Add a New Secret**: Click on `+ New secret`.
3.  **Enter Secret Name**: In the `Name` field, type exactly `COMPUTE_PRICES_API_KEY`.
4.  **Enter Secret Value**: In the `Value` field, paste your API key: `cp_live_nWBy8ZSpnJfXgZzl2UU7xrWRG84sW7KVTYtzEeJgwLo`.
5.  **Save Secret**: Click `Save secret`.
6.  **Enable Notebook Access**: Make sure the toggle next to your newly added secret is set to `ON` for this notebook.

After you've added the secret, please re-run the code cell (`cPFIgbIC2zwe`) above. It should then be able to access the API key and proceed with fetching the data.

# CORRECT SCRIPT FOR GETTING THE DATASET

In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime, UTC # Import UTC for timezone-aware datetime
from google.colab import userdata # Import userdata to access secrets
import sqlite3 # Import sqlite3 for database operations
import time # Import time for scheduling

# API Configuration
BASE_URL = "https://computeprices.com/api/v1/gpu-prices"
DB_NAME = "cloud_gpu_data.db" # Name for the SQLite database

# Retrieve API key securely from Colab Secrets
# IMPORTANT: Ensure a secret named 'COMPUTE_PRICES_API_KEY' is set in Colab Secrets.
# If not set, the script will proceed without an API key, which might lead to errors
# or rate limits if the API requires authentication.
API_KEY = userdata.get("COMPUTE_PRICES_API_KEY")
if API_KEY is None:
    print("Warning: 'COMPUTE_PRICES_API_KEY' not found in Colab Secrets. Proceeding without API key.")
    API_KEY = "" # Default to empty string if secret is not found

def fetch_raw_gpu_data():
    headers = {}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    print("Fetching raw pricing data...")
    try:
        response = requests.get(BASE_URL, headers=headers)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Network error fetching data: {e}")
        return None

def process_and_save():
    payload = fetch_raw_gpu_data()
    if not payload or "data" not in payload:
        print("No data received or invalid payload structure.")
        return None # Return None to indicate failure to process

    records = payload["data"]
    timestamp = datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S") # More standard datetime format

    df = pd.DataFrame(records)

    # Add the extraction timestamp to the DataFrame for historical tracking
    df["extracted_at"] = timestamp

    # Standardize numeric types for data science pipelines
    numeric_cols = ["vram_gb", "price_per_hour_usd", "total_hourly_usd", "gpu_count"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows where critical numeric data might be missing after conversion
    df.dropna(subset=["vram_gb", "price_per_hour_usd", "gpu_count"], inplace=True)

    # Drop columns that have any NaN values (e.g., 'commitment_months')
    df.dropna(axis=1, inplace=True)

    if df.empty:
        print("DataFrame is empty after processing and cleaning. No data to save.")
        return None

    # Save to SQLite database
    try:
        conn = sqlite3.connect(DB_NAME)
        # Use 'gpu_pricing_history' as the table name as suggested by the user
        df.to_sql("gpu_pricing_history", conn, if_exists="append", index=False)
        conn.close()
        print(f"Appended {len(df)} new records to 'gpu_pricing_history' table in {DB_NAME}.")
    except Exception as e:
        print(f"Error saving to SQLite database: {e}")
        return None # Return None if saving to DB fails

    return df

if __name__ == "__main__":
    # Schedule Periodic Pulls
    # Run every 1 to 6 hours (3600 seconds = 1 hour)
    INTERVAL_SECONDS = 300 # Set to 3600 for 1 hour, or 3600 * 6 for 6 hours

    print(f"Starting periodic GPU pricing data fetching (interval: {INTERVAL_SECONDS / 3600} hour(s))...")
    while True:
        try:
            current_df = process_and_save()
            if current_df is not None:
                print(f"Data fetched and saved successfully at {datetime.now(UTC)}.")
                print("\nLatest Dataset Preview:")
                # Display only relevant columns for the preview
                display_cols = ["gpu", "provider", "vram_gb", "price_per_hour_usd", "pricing_type", "extracted_at"]
                # Ensure display_cols only includes columns actually present in current_df
                display_cols_present = [col for col in display_cols if col in current_df.columns]
                print(current_df[display_cols_present].head())
            else:
                print(f"Failed to fetch or process data at {datetime.now(UTC)}.")
        except Exception as e:
            print(f"An unexpected error occurred during the loop: {e}")

        print(f"Waiting for {INTERVAL_SECONDS / 60} minutes before the next fetch...")
        time.sleep(INTERVAL_SECONDS)

Starting periodic GPU pricing data fetching (interval: 0.08333333333333333 hour(s))...
Fetching raw pricing data...
Appended 1042 new records to 'gpu_pricing_history' table in cloud_gpu_data.db.
Data fetched and saved successfully at 2026-07-10 00:03:21.535570+00:00.

Latest Dataset Preview:
        gpu    provider  vram_gb  price_per_hour_usd pricing_type  \
0  A100 SXM  Deep Infra       80                0.89    on_demand   
1  A100 SXM  Deep Infra       80                0.89     reserved   
2  H100 SXM  Deep Infra       80                1.79    on_demand   
3  H100 SXM  Deep Infra       80                1.79     reserved   
4      H200  Deep Infra      141                2.19    on_demand   

          extracted_at  
0  2026-07-10 00:03:21  
1  2026-07-10 00:03:21  
2  2026-07-10 00:03:21  
3  2026-07-10 00:03:21  
4  2026-07-10 00:03:21  
Waiting for 5.0 minutes before the next fetch...
Fetching raw pricing data...
Appended 1042 new records to 'gpu_pricing_history' table in clou

KeyboardInterrupt: 

In [ ]:
import sqlite3
import pandas as pd

# Define the database file name
DB_NAME = "cloud_gpu_data.db"
CSV_OUTPUT_NAME = "gpu_pricing_history.csv"

try:
    # Connect to the SQLite database
    conn = sqlite3.connect(DB_NAME)

    # Read the 'gpu_pricing' table into a pandas DataFrame
    # Note: Replace 'gpu_pricing_history' with your actual table name if different
    df_from_db = pd.read_sql_query("SELECT * FROM gpu_pricing", conn) # Corrected table name

    # Close the database connection
    conn.close()

    print(f"Successfully loaded {len(df_from_db)} records from '{DB_NAME}'.")

    # Save the DataFrame to a CSV file
    df_from_db.to_csv(CSV_OUTPUT_NAME, index=False)
    print(f"Data saved to '{CSV_OUTPUT_NAME}'.")

    # Display the first 5 rows of the loaded DataFrame
    print("\nFirst 5 rows of the loaded data from the database:")
    display(df_from_db.head())

except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded 1044 records from 'cloud_gpu_data.db'.
Data saved to 'gpu_pricing_history.csv'.

First 5 rows of the loaded data from the database:


,gpu_model,provider,vram_gb,hourly_rate_usd,pricing_type,data_source,extracted_at
0,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53
1,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53
2,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53
3,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53
4,H200,Deep Infra,141,2.19,On-Demand,ComputePrices_API,2026-07-08 21:34:53


In [ ]:
len(df_from_db)

1051

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("cloud_gpu_data.db")

df = pd.read_sql_query(
    "SELECT * FROM gpu_pricing_history",
    conn
)

df.to_csv(
    "gpu_pricing_history.csv",
    index=False
)

print(df.head())
print(f"Rows: {len(df)}")

     provider provider_slug           provider_url       gpu gpu_slug  \
0  Deep Infra    deep-infra  https://deepinfra.com  A100 SXM  a100sxm   
1  Deep Infra    deep-infra  https://deepinfra.com  A100 SXM  a100sxm   
2  Deep Infra    deep-infra  https://deepinfra.com  H100 SXM     h100   
3  Deep Infra    deep-infra  https://deepinfra.com  H100 SXM     h100   
4  Deep Infra    deep-infra  https://deepinfra.com      H200     h200   

   vram_gb architecture  gpu_count  max_gpus_per_node  price_per_hour_usd  \
0       80       Ampere          1                  1                0.89   
1       80       Ampere          1                  1                0.89   
2       80       Hopper          1                  1                1.79   
3       80       Hopper          1                  1                1.79   
4      141       Hopper          1                  1                2.19   

   total_hourly_usd pricing_type  commitment_months currency  \
0              0.89    on_demand  

In [ ]:
df.sample(5)

,provider,provider_slug,provider_url,gpu,gpu_slug,vram_gb,architecture,gpu_count,max_gpus_per_node,price_per_hour_usd,total_hourly_usd,pricing_type,commitment_months,currency,exchange_rate_to_usd,source_url,last_updated,extracted_at
2695,Crusoe,crusoe,https://www.crusoe.ai/,A100 SXM,a100sxm,80,Ampere,4,8,1.4500,5.8000,on_demand,NaN,USD,1.0,https://shadeform.com/directory/instances?clou...,2026-07-09T00:00:06.274954+00:00,2026-07-10 00:03:21
1945,Sesterce,sesterce,https://www.sesterce.com/,B200,b200,192,Blackwell,4,8,7.0593,28.2372,on_demand,NaN,USD,1.0,https://docs.sesterce.com/api-reference/gpu-cl...,2026-07-09T00:00:26.245817+00:00,2026-07-09 21:50:49
854,Sesterce,sesterce,https://www.sesterce.com/,L40,l40,40,Ada Lovelace,4,8,1.1000,4.4000,on_demand,NaN,USD,1.0,https://docs.sesterce.com/api-reference/gpu-cl...,2026-07-09T00:00:26.496248+00:00,2026-07-09 21:45:47
3431,Vast.ai,vast,https://vast.ai/,L40S,l40s,48,Ada Lovelace,1,1,0.4681,0.4681,on_demand,NaN,USD,1.0,https://cloud.vast.ai/?ref_id=236521,2026-06-12T00:00:46.053387+00:00,2026-07-10 00:08:22
919,Latitude.sh,latitude,https://www.latitude.sh/,L40S,l40s,48,Ada Lovelace,8,8,0.8713,6.9704,on_demand,NaN,USD,1.0,https://shadeform.com/directory/instances?clou...,2026-07-09T00:00:07.405233+00:00,2026-07-09 21:45:47


In [ ]:
len(df)

1051

In [ ]:
import pandas as pd

# Concatenate the two DataFrames
# Assuming both dataframes have the same columns and represent similar data,
# we will use pd.concat to stack them vertically.
combined_df = pd.concat([df_from_db, df], ignore_index=True)

# Define the output CSV file name
combined_csv_filename = "combined_gpu_data.csv"

# Save the combined DataFrame to a CSV file
combined_df.to_csv(combined_csv_filename, index=False)

print(f"Successfully combined {len(df_from_db)} records from df_from_db and {len(df)} records from df_after_db.")
print(f"Total records in combined DataFrame: {len(combined_df)}.")
print(f"Combined data saved to '{combined_csv_filename}'.")

# Display the first few rows of the combined DataFrame
print("\nFirst 5 rows of the combined data:")
display(combined_df.head())

Successfully combined 1044 records from df_from_db and 1051 records from df_after_db.
Total records in combined DataFrame: 2095.
Combined data saved to 'combined_gpu_data.csv'.

First 5 rows of the combined data:


,gpu_model,provider,vram_gb,hourly_rate_usd,pricing_type,data_source,extracted_at,provider_slug,provider_url,gpu,gpu_slug,architecture,gpu_count,max_gpus_per_node,price_per_hour_usd,total_hourly_usd,currency,exchange_rate_to_usd,source_url,last_updated
0,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,H200,Deep Infra,141,2.19,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
len(combined_df)

2095

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1051 entries, 0 to 1050
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   provider              1051 non-null   object 
 1   provider_slug         1051 non-null   object 
 2   provider_url          1051 non-null   object 
 3   gpu                   1051 non-null   object 
 4   gpu_slug              1051 non-null   object 
 5   vram_gb               1051 non-null   int64  
 6   architecture          1051 non-null   object 
 7   gpu_count             1051 non-null   int64  
 8   max_gpus_per_node     1051 non-null   int64  
 9   price_per_hour_usd    1051 non-null   float64
 10  total_hourly_usd      1051 non-null   float64
 11  pricing_type          1051 non-null   object 
 12  commitment_months     152 non-null    float64
 13  currency              1051 non-null   object 
 14  exchange_rate_to_usd  1051 non-null   float64
 15  source_url           

In [ ]:
df.isnull().sum()

,0
provider,0
provider_slug,0
provider_url,0
gpu,0
gpu_slug,0
vram_gb,0
architecture,0
gpu_count,0
max_gpus_per_node,0
price_per_hour_usd,0


In [ ]:
len(df)

1051

In [ ]:
print("Original NaN count per column:")
display(combined_df.isnull().sum())

# Drop columns that have any NaN values
# The 'inplace=True' argument modifies the DataFrame directly
combined_df.dropna(axis=1, inplace=True)

print("\nDataFrame after dropping columns with NaN values:")
print("Updated NaN count per column:")
display(combined_df.isnull().sum())

print("\nFirst 5 rows of the updated DataFrame:")
display(combined_df.head())


Original NaN count per column:


,0
gpu_model,1051
provider,0
vram_gb,0
hourly_rate_usd,1051
pricing_type,0
data_source,1051
extracted_at,0
provider_slug,1044
provider_url,1044
gpu,1044



DataFrame after dropping columns with NaN values:
Updated NaN count per column:


,0
provider,0
vram_gb,0
pricing_type,0
extracted_at,0



First 5 rows of the updated DataFrame:


,provider,vram_gb,pricing_type,extracted_at
0,Deep Infra,80,On-Demand,2026-07-08 21:34:53
1,Deep Infra,80,On-Demand,2026-07-08 21:34:53
2,Deep Infra,80,On-Demand,2026-07-08 21:34:53
3,Deep Infra,80,On-Demand,2026-07-08 21:34:53
4,Deep Infra,141,On-Demand,2026-07-08 21:34:53


In [ ]:
len(df)

1051

In [ ]:
df.isnull().sum()

,0
provider,0
provider_slug,0
provider_url,0
gpu,0
gpu_slug,0
vram_gb,0
architecture,0
gpu_count,0
max_gpus_per_node,0
price_per_hour_usd,0


In [ ]:
df_from_db.isnull().sum()

,0
gpu_model,0
provider,0
vram_gb,0
hourly_rate_usd,0
pricing_type,0
data_source,0
extracted_at,0


In [ ]:
df.sample(1)

,provider,provider_slug,provider_url,gpu,gpu_slug,vram_gb,architecture,gpu_count,max_gpus_per_node,price_per_hour_usd,total_hourly_usd,pricing_type,currency,exchange_rate_to_usd,source_url,last_updated,extracted_at
543,Together AI,together-ai,https://www.together.ai,A100 SXM,a100sxm,80,Ampere,1,8,2.592,2.592,on_demand,USD,1.0,https://www.together.ai/pricing,2026-07-09T00:00:58.940278+00:00,2026-07-09 20:55:57


In [ ]:
df_from_db.sample(1)

,gpu_model,provider,vram_gb,hourly_rate_usd,pricing_type,data_source,extracted_at
875,L40S,Sesterce,48,0.968,On-Demand,ComputePrices_API,2026-07-08 21:34:53
